<a href="https://colab.research.google.com/github/charansai12322/bffdm/blob/main/bffdm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset

dataset = load_dataset("CiferAI/Cifer-Fraud-Detection-Dataset-AF", split="train")
print(f"Total transactions: {len(dataset):,}")

Total transactions: 21,000,000


In [ ]:
dataset = dataset.select(range(min(1_000_000, len(dataset))))
print(f"Working with: {len(dataset):,} transactions")

Working with: 1,000,000 transactions


In [ ]:
fraud_data = dataset.filter(lambda x: x['isFraud'] == 1)
non_fraud_data = dataset.filter(lambda x: x['isFraud'] == 0)

print(f"Fraud transactions: {len(fraud_data):,}")
print(f"Non-fraud transactions: {len(non_fraud_data):,}")


Fraud transactions: 1,313
Non-fraud transactions: 998,687


In [ ]:
fraud_sample = fraud_data.shuffle(seed=42).select(range(500))
non_fraud_sample = non_fraud_data.shuffle(seed=42).select(range(500))
from datasets import concatenate_datasets

balanced_dataset = concatenate_datasets([fraud_sample, non_fraud_sample])
balanced_dataset = balanced_dataset.shuffle(seed=42)

print(f"Final training set: {len(balanced_dataset)} transactions")

Final training set: 1000 transactions


In [ ]:
{
    "messages": [
        { "role": "user",      "content": "Your input here" },
        { "role": "assistant", "content": "Expected output here" }
    ]
}

{'messages': [{'role': 'user', 'content': 'Your input here'},
  {'role': 'assistant', 'content': 'Expected output here'}]}

In [ ]:
def create_conversation(example):
    txn_type = example['type']
    amount = example['amount']
    old_orig = example['oldbalanceOrg']
    new_orig = example['newbalanceOrig']
    old_dest = example['oldbalanceDest']
    new_dest = example['newbalanceDest']

    user_msg = f"""Analyze this transaction for fraud risk:
- Type: {txn_type}
- Amount: ${amount:,.2f}
- Sender Balance Before: ${old_orig:,.2f}
- Sender Balance After: ${new_orig:,.2f}
- Recipient Balance Before: ${old_dest:,.2f}
- Recipient Balance After: ${new_dest:,.2f}"""

    if example['isFraud'] == 1:
        asst_msg = "HIGH"
    else:
        asst_msg = "LOW"

    return {"messages": [
        {"role": "user",      "content": user_msg},
        {"role": "assistant", "content": asst_msg}
    ]}

In [ ]:
train_dataset = balanced_dataset.map(create_conversation)
print(f"Created {len(train_dataset)} training conversations")

Created 1000 training conversations


In [ ]:
print(train_dataset[0]['messages'])

[{'content': 'Analyze this transaction for fraud risk:\n- Type: CASH_IN\n- Amount: $243.74\n- Sender Balance Before: $16,101.76\n- Sender Balance After: $11,824,895.38\n- Recipient Balance Before: $1,028,631.69\n- Recipient Balance After: $8.45', 'role': 'user'}, {'content': 'LOW', 'role': 'assistant'}]


In [ ]:
!pip install -q transformers bitsandbytes

In [ ]:
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
!pip install -q accelerate

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print(f"Model loaded: {model_id}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct


In [ ]:
!pip install -q peft

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
!pip install -q datasets trl

In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./fraud-detector",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    max_length=512
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=training_args
)

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())
print("Model dtype:", next(model.parameters()).dtype)

GPU: Tesla T4
BF16 supported: True
Model dtype: torch.bfloat16


In [ ]:
trainer.train()

RuntimeError: gemm input type at::BFloat16 and output type float is only supported for CUDA devices with compute capability 8.0 or higher

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
repo_name = "charansai12322/bffdm"

trainer.push_to_hub(repo_name)

print(f"Uploaded to: https://huggingface.co/{repo_name}")

In [ ]:
print(trainer)

In [ ]:
print(trainer.args.output_dir)

In [ ]:
import os

print(os.listdir(trainer.args.output_dir))

In [ ]:
trainer.save_model("./fraud-detector")

In [ ]:
trainer.push_to_hub("charansai12322/bffdm")

In [ ]:
!pip install -q --upgrade torchao

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model=repo_name, device="cuda")

In [ ]:
question = """Analyze this transaction for fraud risk:
- Type: TRANSFER
- Amount: $85,000.00
- Sender Balance Before: $85,000.00
- Sender Balance After: $0.00
- Recipient Balance Before: $0.00
- Recipient Balance After: $0.00"""

output = generator([{"role": "user", "content": question}], max_new_tokens=150, return_full_text=False)[0]
print(output["generated_text"])